# Exploratory Data Analysis

Describe the full prepared cohort for the Any Readmission project. Each row represents one eligible inpatient encounter selected for one patient. This notebook uses the full cohort for descriptive summaries.

## 1. Setup

Load the prepared cohort and the intermediate encounter tables after checking them against the preparation manifest. Encounter and patient identifiers support integrity checks. The notebook displays aggregate results.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.eda.data import load_eda_inputs

eda = load_eda_inputs(project_root=project_root)

## 2. Cohort Overview

The preparation manifest records the number of encounters and unique patients at each stage. Cleaning preserves all raw rows. Cohort selection excludes encounters with the configured death and hospice discharge codes, then keeps the smallest eligible numeric encounter ID for each patient.

In [ ]:
print(f"Prepared cohort: {len(eda.cohort):,} rows and {eda.cohort.shape[1]} columns.")
print(f"Retained encounters with unclear discharge descriptions: {eda.unclear_discharge_count:,}.")
display(eda.cohort_flow)

**Observation.** The final stage has one row per patient. The cohort also includes encounters with unclear discharge descriptions.

**Limitation.** The smallest eligible ID identifies a representative encounter. Encounter timestamps are unavailable for establishing whether it was the first admission.

### Discharge Eligibility

The encounter table reports the distribution of age, admission type code, and length of stay within each eligibility group. The patient table counts each person once across three mutually exclusive groups. Each percentage divides the row count by the stated denominator. A small cell contains fewer than 30 encounters or patients, according to the table's unit.

In [ ]:
from src.eda.cohort_selection import profile_cohort_selection

selection = profile_cohort_selection(eda)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(selection.discharge_eligibility_profile.round({"percent_of_group": 2}))
    display(selection.discharge_patient_overlap.round({"percent_of_cleaned_patients": 2}))

**Observation.** Ages 80 to under 90 account for 763 of 2,423 excluded encounters (31.49%) and 16,434 of 99,343 eligible encounters (16.54%). Of the patients with an excluded encounter, 871 also have an eligible encounter and 1,528 have only excluded encounters.

**Implication.** The discharge filter changes the age composition of the encounter pool. Patients with an excluded encounter can still contribute an eligible representative encounter.

### Representative Encounter Selection

The distribution counts all patients with at least one eligible encounter. The comparison then focuses on patients with multiple eligible encounters and uses the total encounters in each selection group as the denominator.

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(selection.eligible_encounters_per_patient.round({"percent_of_eligible_patients": 2}))
    display(selection.representative_encounter_profile.round({"percent_of_group": 2}))

**Observation.** The 16,341 patients with multiple eligible encounters contribute 45,694 encounters. They contribute 16,341 representative encounters and 29,353 other eligible encounters. One-day stays account for 2,048 of 16,341 representative encounters (12.53%) and 3,354 of 29,353 other eligible encounters (11.43%).

**Limitation.** A patient contributes one representative encounter and may contribute several other eligible encounters. The percentages describe encounters and the comparison gives more weight to patients with more records in the latter group.

## 3. Data Dictionary

Columns are listed in prepared CSV order. The period describes the information represented by the field. The original three-category outcome remains in the cohort. Field definitions draw on [Strack et al. (2014), Table 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/#tab1).

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(eda.column_dictionary)

## 4. Source Feature Decisions

The preparation audit identifies dropped source columns, replaced diagnosis codes, and derived fields that were considered but not created. Each reason is carried over from the approved preparation step.

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(eda.source_feature_decisions)

## 5. Admission Code Descriptions

Admission Type and Admission Source use separate code lists. A code with an unclear description is recorded in the lookup. An unmapped code is absent from that lookup. The table shows the codes observed in the cohort.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(eda.admission_descriptions.loc[eda.admission_descriptions["observed"]])

## 6. Data Completeness

An empty CSV field is counted separately from the source categories `None` in laboratory results and `Missing` or `Unmapped` in diagnosis groups. Admission codes with unclear descriptions are also reported separately. The overall completeness tables use the full prepared cohort as the denominator.

In [ ]:
from src.eda.descriptive import profile_completeness, profile_numeric

completeness = profile_completeness(eda.cohort, eda.admission_descriptions)
print(
    f"{completeness.empty_field_count:,} empty fields across "
    f"{completeness.patients_with_empty_fields:,} patients."
)
display(completeness.missing_summary.round({"empty_percent": 4}))
display(completeness.missing_patterns.round({"percent": 4}))

In [ ]:
display(completeness.special_values.round({"percent": 4}))
display(completeness.lab_recording_patterns.round({"percent": 4}))

**Observation.** The empty fields occur in race and gender. The laboratory columns have no empty fields, although most patients have no recorded result for either test.

**Implication.** The proportion of empty fields describes the CSV structure. The laboratory and diagnosis tables describe which source information is available for interpretation.

**Limitation.** Literal `None` indicates that no result was recorded. A blank or unexpected value gives an unknown recording status. The source does not establish whether a test was performed. An admission code with an unclear description remains an observed source code.

### Laboratory Recording by History and Patient Characteristics

Prior history refers to recorded outpatient, emergency, and inpatient visits in the year before the representative encounter. History is unknown when any count is missing. For the two laboratory fields, a known result level indicates a recorded result and the literal `None` indicates no recorded result. An empty or unexpected laboratory value makes the combined recording pattern unknown.

Each percentage uses the number of patients in the corresponding group. Blank percentages indicate a group with no patients. A small cell contains fewer than 30 patients. The characteristic table shows age bands, admission type and source codes, and exact days in hospital.

In [ ]:
from src.eda.information_coverage import profile_information_coverage

coverage = profile_information_coverage(eda.cohort)
by_characteristic = coverage.lab_recording_by_characteristic.set_index(
    ["characteristic", "value", "group_patients", "lab_recording_pattern"]
).unstack("lab_recording_pattern", sort=False)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(coverage.lab_recording_by_prior_history.round({"percent_of_group": 2}))
    display(by_characteristic.round(2))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from src.eda.plots import plot_information_coverage

coverage_figures = plot_information_coverage(coverage, eda.admission_descriptions)
for figure in coverage_figures.values():
    plt.close(figure)
    display(figure)

**Observation.** Of the 51,784 patients with zero recorded visits across all three services, 39,726 (76.71%) have neither laboratory result recorded. The corresponding share among patients with at least one recorded visit is 14,315 of 18,206 (78.63%). Among the three largest admission types, this share ranges from 76.91% for Emergency (27,289 of 35,480) to 89.54% for Elective (12,345 of 13,787). Admission type code `6`, with lookup description `NULL`, accounts for 215 of the 246 patients with both results recorded (87.40%). Within code `6`, both results are recorded for 215 of 4,516 patients (4.76%).

**Implication.** The 39,726 patients with zero visits and neither laboratory result recorded form a group for examining which other encounter attributes are available.

**Limitation.** The counts reflect what was recorded in the source. Admission types differ in size and include codes with unclear descriptions. Interpret their shares alongside the group counts.

## 7. Numeric Distributions

The profiles cover all eight numeric predictors. Percentiles use linear interpolation among nonempty observations. The denominator for the zero percentage is the valid count. The plots show the full observed range.

In [ ]:
numeric = profile_numeric(eda.cohort)

with pd.option_context("display.max_columns", None):
    display(numeric.numeric_profile.round({"zero_percent_of_valid": 2}))

In [ ]:
from src.eda.plots import plot_numeric_distributions

figures = plot_numeric_distributions(numeric.numeric_frequency, cohort_size=len(eda.cohort))
for figure in figures.values():
    plt.close(figure)
    display(figure)

### Prior Visits

The positive count for each service type is the denominator for its conditional percentiles. The second table counts the number of service types with at least one recorded visit for each patient. An empty group label represents an unknown count when a service field is missing.

In [ ]:
display(numeric.prior_visit_positive_profile)
display(numeric.prior_visit_coverage.round({"percent": 4}))

**Observation.** The median is zero for each prior service count. Among patients with positive counts, the values extend well above the median. As an example, the maximum outpatient count is above its positive-group 99th percentile. For `number_diagnoses`, 30,719 of 69,990 patients (43.89%) have exactly nine diagnoses recorded. The range from 10 to 16 accounts for 73 patients (0.10%).

**Implication.** The cohort-wide zero share and the positive-only distribution describe different parts of the same variable. The logarithmic count axis makes the rare high values visible. Compare group sizes before analyzing patients with high diagnosis counts. Counts above nine show that the concentration at nine does not establish a recording limit.

**Limitation.** The visit counts describe recorded service use in the year before the representative encounter. High values are retained as observed. Their magnitude alone is insufficient evidence of a data error.

## 8. Nominal Distributions

This section covers all 24 nominal predictors. Counts and percentages use the full prepared cohort unless a table specifies another denominator. An empty CSV field is counted separately. A rare level is an observed nonempty category present in less than 1% of the cohort. The threshold is used to describe frequencies.

In [ ]:
from src.eda.nominal import DEMOGRAPHIC_COLUMNS, MEDICATION_FLAG_COLUMNS, profile_nominal
from src.eda.plots import plot_nominal_distributions

nominal = profile_nominal(eda)
nominal_figures = plot_nominal_distributions(nominal)
for figure in nominal_figures.values():
    plt.close(figure)

display(nominal.nominal_profile.round({"dominant_percent_of_cohort": 4}))

### Demographics

Age is shown in ascending age bands. The race and gender tables include the empty fields in their cohort totals.

In [ ]:
demographics = nominal.nominal_frequency.loc[
    nominal.nominal_frequency["column"].isin(DEMOGRAPHIC_COLUMNS)
]
display(demographics.round({"percent": 4}))
display(nominal_figures["demographic_distributions"])

**Observation.** Ages 70 to under 80 form the largest recorded age band. The recorded category `Caucasian` accounts for most cohort members.

**Limitation.** These distributions describe the prepared cohort. The age bands do not establish exact ages.

### Admission Type and Source

Codes are interpreted within their own lookup. Brown bars show codes without a clear description. The table retains the full descriptions where chart labels are shortened.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(nominal.admission_descriptions.round({"percent": 4}))
display(nominal_figures["admission_distributions"])

**Observation.** `Emergency` is the most frequent admission type and `Emergency Room` is the most frequent admission source. Both fields contain observed codes with unclear descriptions, including `NULL`.

**Implication.** The two fields describe different aspects of the encounter. Code numbers are meaningful only within their respective lookup.

### Laboratory Results

The coverage percentage uses all patients. The percentage for a recorded level within its test uses only patients with a recorded result for that test. A blank conditional percentage indicates that the level is outside that denominator.

In [ ]:
with pd.option_context("display.max_columns", None):
    display(nominal.laboratory_results.round({
        "percent": 4, "recorded_percent_of_cohort": 4, "percent_of_recorded": 4,
    }))
display(nominal_figures["laboratory_result_distributions"])

**Observation.** The cohort has recorded HbA1c results for more patients than recorded serum glucose results. `>8` is the largest recorded HbA1c category, while `Norm` is the largest recorded serum glucose category.

**Limitation.** A conditional result distribution describes patients with a result recorded for that test. Source category labels are preserved as supplied.

### Diabetes Medications

The matrix covers the 12 retained medication columns. A zero means that the given status is absent in the prepared cohort. The non-`No` count includes observed nonempty statuses. The chart identifies small counts on a logarithmic scale above one. `change` and `diabetesMed` are source summary fields and are shown separately.

In [ ]:
medication_counts = nominal.medication_status
count_columns = [name for name in medication_counts.columns if name.endswith("_patients")]
percent_columns = [name for name in medication_counts.columns if name.endswith("_percent")]
with pd.option_context("display.max_columns", None):
    display(medication_counts[["column", "cohort_size", *count_columns]])
    display(medication_counts[["column", "cohort_size", *percent_columns]].round(4))
flags = nominal.nominal_frequency.loc[nominal.nominal_frequency["column"].isin(MEDICATION_FLAG_COLUMNS)]
display(flags.round({"percent": 4}))
display(nominal_figures["medication_distributions"])

**Observation.** Insulin has the most patients with a recorded status other than `No` among the retained medication fields. Some statuses are very rare. For example, only one patient has `Down` for chlorpropamide.

**Limitation.** These tables describe individual recorded medication fields. Counts across drugs can overlap because each patient can have multiple medication records.

### Diagnosis Groups

The same ICD-9 research groups appear in the same order across the three diagnosis positions. The any-position counts include each patient once per group. Patients can appear in multiple groups. The coverage table excludes empty, `Missing` and `Unmapped` values from the number of positions with mapped codes.

In [ ]:
display(nominal.diagnosis_summary.round({"percent_at_any_position": 4}))
display(nominal.diagnosis_coverage.round({"percent": 4}))
display(nominal_figures["diagnosis_group_distributions"])

**Observation.** `Circulatory` is the most frequent group at each diagnosis position. Its count at any position is smaller than the sum of its three position counts because some patients have this group at more than one position.

**Limitation.** A group summarizes several possible source codes. The per-group counts describe coded diagnoses at the selected encounter.

### Diagnosis Positions and Group Co-occurrence

The depth table counts mapped positions and distinct mapped groups for each patient. It also reports `Missing` and `Unmapped` positions separately. The chart compares the first position with any of the three positions. A patient contributes once per group or unordered pair of distinct mapped groups. Percentages in these tables use the full cohort.

In [ ]:
from src.eda.diagnosis_detail import profile_diagnosis_detail
from src.eda.plots import plot_diagnosis_group_gain

diagnoses = profile_diagnosis_detail(eda.cohort, eda.source_cohort, nominal.diagnosis_summary)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(diagnoses.diagnosis_position_depth.round(2))
    display(diagnoses.diagnosis_group_gain.round(2))
    display(diagnoses.diagnosis_group_pairs.round(2))
diagnosis_gain_figure = plot_diagnosis_group_gain(diagnoses.diagnosis_group_gain)
plt.close(diagnosis_gain_figure)
display(diagnosis_gain_figure)

### Source Codes Within Diagnosis Groups

Each nonempty source code is kept as text and counted once per patient. The denominator is the number of patients with that group at any position. A patient can contribute more than one code to a group, so its code percentages can sum to more than 100%. The notebook shows up to five common codes per group. The EDA export includes every observed nonempty source code.

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(diagnoses.diagnosis_source_code_summary.round(2))
    display(diagnoses.diagnosis_source_code_detail.groupby("group").head(5).round(2))

**Observation.** Among 68,706 patients with all three positions mapped, 7,360 have one distinct group, 31,070 have two, and 30,276 have three. The `Diabetes` group appears in the first position for 5,748 patients and at any of the three positions for 27,578 patients. Including the two additional positions adds 21,830 patients, or 31.19 percentage points of the cohort. The Circulatory group combines 57 distinct source codes.

**Implication.** The additional positions increase coverage for many groups. The source-code table shows the detail combined by each broad group and helps define which condition to examine in a later analysis.

**Limitation.** The groups summarize codes recorded at the representative encounter. Multiple positions in one group can contain different codes, and co-occurrence describes shared records in this cohort. The `Diabetes` gain measures the coverage of diagnosis codes in a cohort of patients with diabetes.

## 9. Relationships Between Attributes

These comparisons describe the full prepared cohort. They use attributes from the representative encounter and the year before it. Subgroup counts provide context for comparing distributions and percentages.

### Admission Type and Source

The matrix shows patient counts for every combination of observed admission type and source codes, including zero counts. The heatmap divides each cell by the count for its admission type. Read each row alongside its patient count. Tables retain the full descriptions and the lookup statuses of codes with unclear descriptions.

In [ ]:
from src.eda.admission_context import profile_admission_context
from src.eda.plots import plot_admission_type_source

admission = profile_admission_context(eda.cohort, nominal.admission_descriptions)
pair_counts = admission.admission_type_source.set_index(
    ["admission_type_code", "type_patients", "admission_source_code"]
)["patients"].unstack("admission_source_code", sort=False)
with pd.option_context("display.max_columns", None):
    display(pair_counts)
admission_heatmap = plot_admission_type_source(admission.admission_type_source)
plt.close(admission_heatmap)
display(admission_heatmap)

### Patient Profiles by Admission Code

Age shares use the full patient count for the corresponding admission code. The count summaries report the valid and missing measurements along with the median, quartiles, and zero share among valid measurements. The exported tables flag age cells and valid numeric groups with fewer than 30 patients. The next section describes services by exact days in hospital.

In [ ]:
age_shares = admission.age_by_admission.set_index(
    ["column", "code", "group_patients", "age"]
)["percent_of_group"].unstack("age", sort=False)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(age_shares.round(2))
    display(admission.numeric_by_admission.round({"zero_percent_of_valid": 2}))

**Observation.** The Emergency Room is the source for 31,456 of 35,480 Emergency admissions (88.66%). Physician Referral is the source for 11,934 of 13,787 Elective admissions (86.56%). The median number of non-laboratory procedures is 0 in Emergency admissions and 2 in Elective admissions.

**Implication.** Admission type and source overlap for many encounters. The tables provide the group sizes and service distributions needed to interpret any later comparison by admission context.

**Limitation.** The shares describe the recorded codes in this cohort. Some codes have few patients or unclear descriptions, and the comparisons do not account for differences in patient characteristics.

### Length of Stay and Services

For each exact length of stay, the table reports patient counts, valid measurements, the median, and the interval from Q1 to Q3. Quartiles use linear interpolation among valid measurements. Each stay group appears once per service measure.

In [ ]:
from src.eda.plots import plot_relationships
from src.eda.relationships import profile_relationships

relationships = profile_relationships(
    eda.cohort, target_config_path=project_root / "config" / "target_mapping.json",
)
relationship_figures = plot_relationships(relationships)
for figure in relationship_figures.values():
    plt.close(figure)

display(relationships.stay_service_summary)
display(relationship_figures["stay_service_distributions"])

**Observation.** One-day stays have a median of 10 recorded medications (n = 10,470). For 14-day stays the median is 24 (n = 642). The Q1 to Q3 ranges show variation among patients with the same stay length.

**Limitation.** Length of stay and these measures summarize the same encounter. The contrast does not establish whether service use caused a longer stay or whether the two share other explanations.

### Age and Prior Visits

For each age band and service type, the numerator is the number of patients with at least one recorded visit in the prior year. The denominator is the total patient count in that age band. The valid count shows how many patients have a recorded visit count, including zeros.

In [ ]:
display(relationships.age_prior_visits.round({"percent_of_age_group": 4}))
display(relationship_figures["age_prior_visit_distributions"])

**Observation.** The share with a recorded emergency visit is highest in the 20 to under 30 age band. The share with a recorded inpatient visit is highest in the 80 to under 90 band.

**Limitation.** These age bands differ in size. The percentages describe recorded visits in this cohort and do not establish differences in access to care.

### Retained Medication Fields and Source Flags

The `diabetesMed` comparison counts retained medication fields with a recorded status other than `No`. The `change` comparison uses an indicator for whether any retained medication has status `Up` or `Down`. The table reports percentages both within each source flag group and within the full cohort. Each comparison covers the same patients.

In [ ]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    for flag in ("diabetesMed", "change"):
        rows = relationships.medication_flags.loc[relationships.medication_flags["source_flag"].eq(flag)]
        display(rows.round({"percent_of_flag_group": 4, "percent_of_cohort": 4}))

**Observation.** Among patients with `diabetesMed = Yes`, 34 of 53,304 have no recorded non-`No` status in the retained fields. Among patients with `change = Ch`, 14,247 of 31,497 have no recorded `Up` or `Down` in those fields.

**Limitation.** The comparison covers the 12 retained medication fields. The source flags have broader definitions than these summaries. The observed combinations alone do not establish a recording error.

### Medication Counts, Insulin Combinations, and Excluded Fields

The first table counts the 12 retained fields with `Steady`, `Up`, or `Down` per patient and reports a separate distribution for `Up` or `Down`. The insulin table counts each patient once per insulin and medication pair. Pair percentages use either patients with a recorded insulin status or the full cohort, as labeled. Patients can contribute to multiple pairs. Empty and unexpected statuses are tracked separately from the recorded counts.

The last table follows up the flag groups above using the 11 excluded source medication fields for the same encounters. Each comparison percentage uses the number of patients without the specified retained status within the flag group. An excluded status is unknown when no matching status was found and at least one excluded field has an empty or unexpected value. A flag marks cells with fewer than 30 patients.

In [ ]:
from src.eda.medication_detail import profile_medication_detail

medications = profile_medication_detail(eda.cohort, eda.source_cohort)
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(medications.medication_count_distribution.round(2))
    display(medications.insulin_medication_pairs.round(2))
    display(medications.medication_flag_reconciliation.round(2))

**Observation.** Metformin is the most common retained medication recorded alongside insulin (7,298 of 35,722 patients with insulin recorded, 20.43%). All 34 patients with `diabetesMed = Yes` and no recorded non-`No` status in the retained fields have a matching status in the excluded fields. Among the 14,247 patients with `change = Ch` and no retained `Up` or `Down` status, one has a matching excluded status.

**Implication.** The excluded columns contain medication records for the 34 patients in the `diabetesMed` comparison. They add little coverage of recorded `Up` or `Down` statuses in the `change` comparison.

**Limitation.** These counts compare the information captured by different source fields. The table does not establish why a flag was set or the timing of a medication change.

## 10. Readmission Outcome

The prepared CSV retains `NO`, `<30`, and `>30`. The configured Any Readmission target maps both recorded readmission categories to `readmitted` and maps `NO` to `not_readmitted`. Each representation describes the same full cohort. The binary target is computed in memory.

In [ ]:
display(relationships.outcome_distribution.round({"percent": 4}))
display(relationship_figures["outcome_distributions"])

**Observation.** Recorded readmission accounts for 28,511 of 69,990 patients (40.74%) after combining the two source timing categories.

**Limitation.** `NO` means no readmission was recorded in the source. The source reports timing categories and supplies no exact time to readmission.

## 11. Findings and Open Questions

The evidence below comes from the full prepared cohort. The source table names identify the CSV outputs generated by the EDA export command. The open questions provide context for a later analysis.

In [ ]:
from src.eda.findings import summarize_findings

findings = summarize_findings(eda, completeness, numeric, nominal, relationships)
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(findings)